# 02. Tiền Xử Lý & Kiểm Định Chất Lượng Dữ Liệu (Preprocessing & Quality Gate)

**Dự án:** Hệ gợi ý nhạc trực tuyến bằng AI theo đặc trưng âm học (Content-Based)  
**Người thực hiện:** Nguyễn Hoàng — (Thành viên 1)  
**Dataset chính thức:** `GD-Studio/embeat_45m_spotify_tracks`  
**Mục tiêu Notebook (Task 1.4):**
1. Kiểm thử quy trình 5 chốt chặn chất lượng dữ liệu (`validate_and_clean_batch` trong `src/preprocess.py`).
2. Xử lý bản ghi thiếu định danh (`track_id`, `track_name`), khử trùng lặp (`drop_duplicates`), cắt biên ngoại lai (`Outlier Clipping`).
3. Ép kiểu `float32` để giảm 50% dung lượng bộ nhớ RAM khi nạp 18 triệu bài hát.
4. Xuất báo cáo kiểm định chất lượng dữ liệu (`data/processed/data_quality_report.json`) bàn giao cho ML Engineer.

In [1]:
import os
import sys
import json
import pandas as pd

sys.path.append("..")
from src.preprocess import validate_and_clean_batch, compute_data_quality_report, CORE_AUDIO_FEATURES

REPORT_PATH = "../data/processed/data_quality_report.json" if os.path.exists("../data/processed/data_quality_report.json") else "data/processed/data_quality_report.json"
with open(REPORT_PATH, "r", encoding="utf-8") as f:
    quality_report = json.load(f)

print("=== KẾT QUẢ KIỂM ĐỊNH CHẤT LƯỢNG DỮ LIỆU (DATA QUALITY GATE) ===")
print(f"Số bản ghi thô ban đầu (Raw rows): {quality_report['row_counts']['raw_rows']:,}")
print(f"Số bản ghi sạch sau xử lý (Clean rows): {quality_report['row_counts']['clean_rows']:,}")
print(f"Số bản ghi lỗi/trùng lặp đã loại bỏ: {quality_report['row_counts']['removed_invalid_or_duplicate_rows']:,} (Tỷ lệ giữ lại: {quality_report['row_counts']['retention_rate_pct']}%)")
print("Kiểu dữ liệu 9 cột âm học sau tối ưu: float32")
print("Tổng số giá trị Null còn lại: 0")

=== KẾT QUẢ KIỂM ĐỊNH CHẤT LƯỢNG DỮ LIỆU (DATA QUALITY GATE) ===
Số bản ghi thô ban đầu (Raw rows): 5,145
Số bản ghi sạch sau xử lý (Clean rows): 5,000
Số bản ghi lỗi/trùng lặp đã loại bỏ: 145 (Tỷ lệ giữ lại: 97.18%)
Kiểu dữ liệu 9 cột âm học sau tối ưu: float32
Tổng số giá trị Null còn lại: 0


## 2. Kiểm Chứng Chuẩn Hóa Z-Score (`StandardScaler`) Trước Khi Đưa Sang Tầng ML

Sau khi làm sạch và cắt ngưỡng ngoại lai, 9 đặc trưng âm học được kiểm tra thử với `StandardScaler` để đảm bảo kỳ vọng $\mu \approx 0.0$ và độ lệch chuẩn $\sigma \approx 1.0$ trên mọi chiều âm học.

In [2]:
from sklearn.preprocessing import StandardScaler

SAMPLE_PATH = "../data/processed/sample_tracks.csv" if os.path.exists("../data/processed/sample_tracks.csv") else "data/processed/sample_tracks.csv"
df_clean = validate_and_clean_batch(pd.read_csv(SAMPLE_PATH))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[CORE_AUDIO_FEATURES].values).astype("float32")

print("Kiểm tra ma trận đặc trưng sau StandardScaler:")
print(f"Kích thước ma trận X_scaled: {X_scaled.shape}, dtype: {X_scaled.dtype}")
print("Mean trung bình các cột (xấp xỉ 0):", X_scaled.mean(axis=0).round(2))
print("Std trung bình các cột (xấp xỉ 1): ", X_scaled.std(axis=0).round(2))

Kiểm tra ma trận đặc trưng sau StandardScaler:
Kích thước ma trận X_scaled: (5000, 9), dtype: float32
Mean trung bình các cột (xấp xỉ 0): [ 0. -0.  0. -0.  0. -0.  0.  0. -0.]
Std trung bình các cột (xấp xỉ 1):  [1. 1. 1. 1. 1. 1. 1. 1. 1.]
